In [1]:
import json, csv
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES  = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]
SR       = 16000
TIME_FRAMES = 128
BATCH    = 32
LR       = 5e-4
EPOCHS   = 40
PATIENCE = 8

with open("audioset_index.json") as f:
    IDX = json.load(f)

print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

Device: cuda | NVIDIA RTX A4000


In [2]:
class AudioSetDS(Dataset):
    def __init__(self, csv_path, n_mels=64, augment=False):
        self.df = pd.read_csv(csv_path)
        self.df["ytid"] = self.df["ytid"].str.strip()
        self.df = self.df[self.df["ytid"].isin(IDX)].reset_index(drop=True)
        self.augment = augment
        self.n_mels = n_mels

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row    = self.df.iloc[i]
        labels = np.array([row[c] for c in CLASSES], dtype=np.float32)
        y, _   = librosa.load(IDX[row["ytid"]], sr=SR, duration=10.0)

        if self.augment:
            if np.random.rand() < 0.5:
                y = y + np.random.randn(len(y)) * 0.005
            if np.random.rand() < 0.4:
                y = librosa.effects.time_stretch(y, rate=np.random.uniform(0.9, 1.1))
            if np.random.rand() < 0.3:
                y = librosa.effects.pitch_shift(y, sr=SR, n_steps=np.random.randint(-2, 3))

        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=self.n_mels))
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        mel = np.pad(mel, ((0,0),(0,TIME_FRAMES-mel.shape[1]))) if mel.shape[1] < TIME_FRAMES else mel[:, :TIME_FRAMES]

        if self.augment:
            if np.random.rand() < 0.5:
                f0 = np.random.randint(0, max(self.n_mels-16,1)); mel[f0:f0+np.random.randint(4,16), :] = 0
            if np.random.rand() < 0.5:
                t0 = np.random.randint(0, TIME_FRAMES-20); mel[:, t0:t0+np.random.randint(5,20)] = 0

        return torch.tensor(mel[np.newaxis], dtype=torch.float32), torch.tensor(labels)

def mil_loss_max(fl, cl, crit):  return crit(fl.max(1).values, cl)
def mil_loss_mean(fl, cl, crit): return crit(fl.mean(1), cl)

print("Dataset class ready.")

Dataset class ready.


In [3]:
class CRNN_v3(nn.Module):
    """Large: 3 blocks, double-conv, 128 mels, 2-layer LSTM."""
    def __init__(self, n, sed=True):
        super().__init__()
        self.sed = sed
        def blk(i,o,pool=(2,2)):
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(pool), nn.Dropout2d(0.1))
        self.cnn  = nn.Sequential(blk(1,32,(2,2)), blk(32,64,(2,2)), blk(64,128,(2,1)))
        self.lstm = nn.LSTM(128*16, 128, batch_first=True, bidirectional=True, num_layers=2, dropout=0.2)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(256, n)
    def forward(self, x):
        x = self.cnn(x); b,c,f,t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b,t,c*f)
        x,_ = self.lstm(x); x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))

class CRNN_v2_128mel(nn.Module):
    """Small v2 architecture, fed 128-mel input instead of 64 — isolates mel resolution."""
    def __init__(self, n, sed=True):
        super().__init__()
        self.sed = sed
        self.cnn = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.lstm = nn.LSTM(32*32, 64, batch_first=True, bidirectional=True)  # 128/4=32 freq bins
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(128, n)
    def forward(self, x):
        x = self.cnn(x); b,c,f,t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b,t,c*f)
        x,_ = self.lstm(x); x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))

print("Models ready.")
print("CRNN_v3 params:", sum(p.numel() for p in CRNN_v3(8).parameters()))
print("CRNN_v2_128mel params:", sum(p.numel() for p in CRNN_v2_128mel(8).parameters()))

Models ready.
CRNN_v3 params: 2914920
CRNN_v2_128mel params: 564008


In [4]:
def train_ablation(model, train_ds, val_ds, mil_fn, tag, epochs=EPOCHS):
    tr = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=8, pin_memory=True)
    va = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=8, pin_memory=True)

    dft = train_ds.df
    pw  = [min((len(dft)-dft[c].sum())/max(dft[c].sum(),1), 50) for c in CLASSES]
    pos_weight = torch.tensor(pw, dtype=torch.float32).to(DEVICE)

    crit   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=3, factor=0.5)
    scaler = torch.amp.GradScaler("cuda")

    best_f1, pc, nb = 0.0, 0, len(tr)
    for ep in range(epochs):
        model.train()
        for bi, (X, y) in enumerate(tr):
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                out = model(X)
                loss = mil_fn(out, y, crit)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            print(f"\r[{tag}] Ep{ep+1:02d} batch {bi+1}/{nb}", end="")

        model.eval(); P_, L_ = [], []
        with torch.no_grad():
            for X, y in va:
                with torch.amp.autocast("cuda"):
                    pr = torch.sigmoid(model(X.to(DEVICE)).max(1).values).float().cpu().numpy()
                P_.append((pr>0.5).astype(int)); L_.append(y.numpy())
        P_, L_ = np.vstack(P_), np.vstack(L_)
        mi = f1_score(L_, P_, average="micro", zero_division=0)
        print(f"\r[{tag}] Ep{ep+1:02d} | val micro-F1 {mi:.4f}                    ")
        sched.step(mi)

        if mi > best_f1:
            best_f1, pc = mi, 0
            torch.save(model.state_dict(), f"ablation_{tag}.pth")
        else:
            pc += 1
            if pc >= PATIENCE:
                print(f"[{tag}] Early stop at ep{ep+1}"); break

    print(f"\n[{tag}] DONE. Best val micro-F1: {best_f1:.4f}")
    return best_f1

In [5]:
ds_train_noaug = AudioSetDS("audioset_v2_train.csv", n_mels=128, augment=False)
ds_val         = AudioSetDS("audioset_v2_val.csv",   n_mels=128, augment=False)

f1_noaug = train_ablation(CRNN_v3(8).to(DEVICE), ds_train_noaug, ds_val, mil_loss_max, "noaug")
print(f"\nNo-aug: {f1_noaug:.4f}   |   v3 with aug: 0.6162")

[noaug] Ep01 batch 32/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[noaug] Ep01 | val micro-F1 0.4490                    
[noaug] Ep02 batch 32/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[noaug] Ep02 batch 200/311

...

[noaug] Ep39 | val micro-F1 0.6312                    
[noaug] Ep40 batch 44/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[noaug] Ep40 batch 179/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[noaug] Ep40 | val micro-F1 0.6371                    

[noaug] DONE. Best val micro-F1: 0.6371

No-aug: 0.6371   |   v3 with aug: 0.6162


In [6]:
ds_train_128 = AudioSetDS("audioset_v2_train.csv", n_mels=128, augment=True)
ds_val_128   = AudioSetDS("audioset_v2_val.csv",   n_mels=128, augment=False)

f1_128small = train_ablation(CRNN_v2_128mel(8).to(DEVICE), ds_train_128, ds_val_128, mil_loss_max, "128mel_small")
print(f"\n128-mel small arch: {f1_128small:.4f}   |   v2(64mel,small): 0.5675   |   v3(128mel,large): 0.6162")

el_small] Ep01 batch 24/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


el_small] Ep01 batch 246/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


el_small] Ep01 | val micro-F1 0.3947                    
el_small] Ep02 batch 8/311

...

el_small] Ep39 batch 281/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


el_small] Ep39 | val micro-F1 0.5648                    
el_small] Ep40 batch 196/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


el_small] Ep40 | val micro-F1 0.5724                    

el_small] DONE. Best val micro-F1: 0.5770

128-mel small arch: 0.5770   |   v2(64mel,small): 0.5675   |   v3(128mel,large): 0.6162


In [7]:
ds_train_v3 = AudioSetDS("audioset_v2_train.csv", n_mels=128, augment=True)

f1_meanpool = train_ablation(CRNN_v3(8).to(DEVICE), ds_train_v3, ds_val_128, mil_loss_mean, "meanpool")
print(f"\nMean-pool: {f1_meanpool:.4f}   |   Max-pool (v3): 0.6162")

[meanpool] Ep01 batch 107/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[meanpool] Ep01 batch 244/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[meanpool] Ep01 | val micro-F1 0.3233                    
[meanpool] Ep02 batch 136/311

...

[meanpool] Ep39 | val micro-F1 0.4609                    
[meanpool] Ep40 batch 131/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[meanpool] Ep40 batch 158/311

/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


[meanpool] Ep40 | val micro-F1 0.4692                    

[meanpool] DONE. Best val micro-F1: 0.4749

Mean-pool: 0.4749   |   Max-pool (v3): 0.6162


In [8]:
model_noaug = CRNN_v3(8, sed=True).to(DEVICE)
model_noaug.load_state_dict(torch.load("ablation_noaug.pth"))
model_noaug.eval()

test_df_ab = pd.read_csv("audioset_v2_test.csv")
test_df_ab["ytid"] = test_df_ab["ytid"].str.strip()
te_ds_ab = AudioSetDS("audioset_v2_test.csv", n_mels=128, augment=False)
te_loader_ab = DataLoader(te_ds_ab, batch_size=BATCH, shuffle=False, num_workers=8)

TP, TL = [], []
with torch.no_grad():
    for X, y in te_loader_ab:
        with torch.amp.autocast("cuda"):
            pr = torch.sigmoid(model_noaug(X.to(DEVICE)).max(1).values).float().cpu().numpy()
        TP.append(pr); TL.append(y.numpy())
test_probs_noaug  = np.vstack(TP).astype(np.float32)
test_labels_noaug = np.vstack(TL).astype(np.float32)

from sklearn.metrics import average_precision_score
thr_grid = np.arange(0.10, 0.95, 0.01)
aps = []
for j, cls in enumerate(CLASSES):
    gt, sc = test_labels_noaug[:,j], test_probs_noaug[:,j]
    ap = average_precision_score(gt, sc) if gt.sum()>0 else 0
    aps.append(ap)

mAP_noaug = np.mean(aps)

best_thr_noaug = {}
tuned = np.zeros_like(test_probs_noaug)
for j, cls in enumerate(CLASSES):
    gt, sc = test_labels_noaug[:,j], test_probs_noaug[:,j]
    bf, bt = 0, 0.5
    for t in thr_grid:
        f = f1_score(gt, (sc>t).astype(int), zero_division=0)
        if f>bf: bf, bt = f, t
    best_thr_noaug[cls] = bt
    tuned[:,j] = (sc > bt).astype(np.float32)

mi_noaug = f1_score(test_labels_noaug, tuned, average="micro", zero_division=0)
ma_noaug = f1_score(test_labels_noaug, tuned, average="macro", zero_division=0)

print(f"=== NO-AUGMENTATION — TEST SET (eval_segments) ===")
print(f"mAP      : {mAP_noaug:.4f}   |  v3 (with aug): 0.634")
print(f"Micro-F1 : {mi_noaug:.4f}   |  v3 (with aug): 0.647")
print(f"Macro-F1 : {ma_noaug:.4f}   |  v3 (with aug): 0.613")

/tmp/ipykernel_41455/611687463.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_noaug.load_state_dict(torch.load("ablation_noaug.pth"))
/user/HS400/as07181/minicond

=== NO-AUGMENTATION — TEST SET (eval_segments) ===
mAP      : 0.6241   |  v3 (with aug): 0.634
Micro-F1 : 0.6420   |  v3 (with aug): 0.647
Macro-F1 : 0.5993   |  v3 (with aug): 0.613


In [9]:
model_128small = CRNN_v2_128mel(8, sed=True).to(DEVICE)
model_128small.load_state_dict(torch.load("ablation_128mel_small.pth"))
model_128small.eval()

te_ds_128 = AudioSetDS("audioset_v2_test.csv", n_mels=128, augment=False)
te_loader_128 = DataLoader(te_ds_128, batch_size=BATCH, shuffle=False, num_workers=8)

TP, TL = [], []
with torch.no_grad():
    for X, y in te_loader_128:
        with torch.amp.autocast("cuda"):
            pr = torch.sigmoid(model_128small(X.to(DEVICE)).max(1).values).float().cpu().numpy()
        TP.append(pr); TL.append(y.numpy())
test_probs_128  = np.vstack(TP).astype(np.float32)
test_labels_128 = np.vstack(TL).astype(np.float32)

from sklearn.metrics import average_precision_score
aps = [average_precision_score(test_labels_128[:,j], test_probs_128[:,j])
       if test_labels_128[:,j].sum()>0 else 0 for j in range(8)]
mAP_128small = np.mean(aps)

thr_grid = np.arange(0.10, 0.95, 0.01)
tuned = np.zeros_like(test_probs_128)
for j in range(8):
    gt, sc = test_labels_128[:,j], test_probs_128[:,j]
    bf, bt = 0, 0.5
    for t in thr_grid:
        f = f1_score(gt, (sc>t).astype(int), zero_division=0)
        if f>bf: bf, bt = f, t
    tuned[:,j] = (sc > bt).astype(np.float32)

mi_128small = f1_score(test_labels_128, tuned, average="micro", zero_division=0)

print(f"=== 128-MEL, SMALL ARCH — TEST SET ===")
print(f"mAP      : {mAP_128small:.4f}   |  v2 (test): 0.535  |  v3 (test): 0.634")
print(f"Micro-F1 : {mi_128small:.4f}   |  v2 (test): 0.587  |  v3 (test): 0.647")

/tmp/ipykernel_41455/3862884957.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_128small.load_state_dict(torch.load("ablation_128mel_small.pth"))
/user/HS400/as071

=== 128-MEL, SMALL ARCH — TEST SET ===
mAP      : 0.5748   |  v2 (test): 0.535  |  v3 (test): 0.634
Micro-F1 : 0.6048   |  v2 (test): 0.587  |  v3 (test): 0.647


In [10]:
model_meanpool = CRNN_v3(8, sed=True).to(DEVICE)
model_meanpool.load_state_dict(torch.load("ablation_meanpool.pth"))
model_meanpool.eval()

te_loader_mp = DataLoader(AudioSetDS("audioset_v2_test.csv", n_mels=128, augment=False),
                          batch_size=BATCH, shuffle=False, num_workers=8)
TP, TL = [], []
with torch.no_grad():
    for X, y in te_loader_mp:
        with torch.amp.autocast("cuda"):
            pr = torch.sigmoid(model_meanpool(X.to(DEVICE)).max(1).values).float().cpu().numpy()
        TP.append(pr); TL.append(y.numpy())
test_probs_mp  = np.vstack(TP).astype(np.float32)
test_labels_mp = np.vstack(TL).astype(np.float32)

aps = [average_precision_score(test_labels_mp[:,j], test_probs_mp[:,j])
       if test_labels_mp[:,j].sum()>0 else 0 for j in range(8)]
mAP_mp = np.mean(aps)

tuned = np.zeros_like(test_probs_mp)
for j in range(8):
    gt, sc = test_labels_mp[:,j], test_probs_mp[:,j]
    bf, bt = 0, 0.5
    for t in thr_grid:
        f = f1_score(gt, (sc>t).astype(int), zero_division=0)
        if f>bf: bf, bt = f, t
    tuned[:,j] = (sc > bt).astype(np.float32)
mi_mp = f1_score(test_labels_mp, tuned, average="micro", zero_division=0)

print(f"=== MEAN-POOL MIL — TEST SET ===")
print(f"mAP      : {mAP_mp:.4f}   |  v3 max-pool (test): 0.634")
print(f"Micro-F1 : {mi_mp:.4f}   |  v3 max-pool (test): 0.647")

/tmp/ipykernel_41455/2845723101.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_meanpool.load_state_dict(torch.load("ablation_meanpool.pth"))
/user/HS400/as07181/m

=== MEAN-POOL MIL — TEST SET ===
mAP      : 0.6044   |  v3 max-pool (test): 0.634
Micro-F1 : 0.6170   |  v3 max-pool (test): 0.647


In [1]:
import json, librosa, numpy as np, pandas as pd, warnings
from collections import Counter

warnings.filterwarnings("ignore")  # suppress the librosa spam while counting

with open("audioset_index.json") as f:
    IDX = json.load(f)

CLASSES = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]
SR = 16000
MIN_SAMPLES = 1600          # 0.1 s at 16 kHz — below this is unusable

results = {}
bad_by_class = Counter()

for split in ["train", "val", "test"]:
    df = pd.read_csv(f"audioset_v2_{split}.csv")
    df["ytid"] = df["ytid"].str.strip()
    df = df[df["ytid"].isin(IDX)].reset_index(drop=True)

    empty, short, failed, ok = 0, 0, 0, 0

    for i, row in df.iterrows():
        try:
            y, _ = librosa.load(IDX[row["ytid"]], sr=SR, duration=10.0)
            n = len(y)
            if n == 0:
                empty += 1
                for c in CLASSES:
                    if row[c] == 1: bad_by_class[c] += 1
            elif n < MIN_SAMPLES:
                short += 1
                for c in CLASSES:
                    if row[c] == 1: bad_by_class[c] += 1
            else:
                ok += 1
        except Exception:
            failed += 1
            for c in CLASSES:
                if row[c] == 1: bad_by_class[c] += 1

        if (i + 1) % 500 == 0:
            print(f"\r[{split}] {i+1}/{len(df)}", end="")

    total = len(df)
    bad = empty + short + failed
    results[split] = dict(total=total, empty=empty, short=short,
                          failed=failed, bad=bad, ok=ok)
    print(f"\r[{split}] {total} clips | empty={empty} short={short} "
          f"failed={failed} | unusable={bad} ({bad/total*100:.2f}%)")

# ---- summary ----
gt = sum(r["total"] for r in results.values())
gb = sum(r["bad"]   for r in results.values())
print("\n" + "="*55)
print(f"TOTAL: {gb} unusable of {gt} clips  ({gb/gt*100:.2f}%)")
print("="*55)

if bad_by_class:
    print("\nUnusable clips by class:")
    for c in CLASSES:
        print(f"  {c:10s}: {bad_by_class[c]}")

json.dump({"per_split": results,
           "total_clips": gt, "total_unusable": gb,
           "pct_unusable": round(gb/gt*100, 3),
           "by_class": dict(bad_by_class)},
          open("results/corrupt_clip_audit.json","w"), indent=2)
print("\n✓ Saved results/corrupt_clip_audit.json")

[train] 9922 clips | empty=2 short=0 failed=0 | unusable=2 (0.02%)
[val] 1752 clips | empty=0 short=0 failed=0 | unusable=0 (0.00%)
[test] 1249 clips | empty=1 short=0 failed=0 | unusable=1 (0.08%)

TOTAL: 3 unusable of 12923 clips  (0.02%)

Unusable clips by class:
  Scream    : 0
  Shout     : 0
  Crying    : 0
  Explosion : 1
  Gunshot   : 1
  Glass     : 0
  Siren     : 0
  Alarm     : 0

✓ Saved results/corrupt_clip_audit.json
